<a href="https://colab.research.google.com/github/Anupampal1992/Group-Project/blob/EDA/Traffic_Accident.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [156]:
!git clone  https://github.com/Anupampal1992/Group-Project.git
%cd Group-Project
!ls

Cloning into 'Group-Project'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 36 (delta 21), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 196.77 KiB | 4.19 MiB/s, done.
Resolving deltas: 100% (21/21), done.
/content/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project/Group-Project
 Traffic_Accident.ipynb  'Traffic Accident Severity Predictor Dataset.csv'


In [157]:

!git status


On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [158]:
import pandas as pd
import numpy as np

#visualization
import matplotlib.pyplot as plt
import seaborn as sns

#preprocessing
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from scipy.spatial import transform


#Models
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

#Evaluation
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score,accuracy_score

#Pipeline
from sklearn.pipeline import Pipeline

#Compute Class Weight
from sklearn.utils.class_weight import compute_class_weight

In [159]:
df=pd.read_csv('Traffic Accident Severity Predictor Dataset.csv')  #Loading the data from Github
df.head(5)

,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
0,Rainy,City Road,Morning,1.0,100.0,5.0,0.0,NaN,Wet,Car,51.0,48.0,Artificial Light,0
1,Clear,Rural Road,Night,NaN,120.0,3.0,0.0,Moderate,Wet,Truck,49.0,43.0,Artificial Light,0
2,Rainy,Highway,Evening,1.0,60.0,4.0,0.0,Low,Icy,Car,54.0,52.0,Artificial Light,0
3,Clear,City Road,Afternoon,2.0,60.0,3.0,0.0,Low,Under Construction,Bus,34.0,31.0,Daylight,0
4,Rainy,Highway,Morning,1.0,195.0,11.0,0.0,Low,Dry,Car,62.0,55.0,Artificial Light,1


EDA

In [160]:
df.shape

(20000, 14)

In [161]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Weather               19958 non-null  object 
 1   Road_Type             19958 non-null  object 
 2   Time_of_Day           19958 non-null  object 
 3   Traffic_Density       19958 non-null  float64
 4   Speed_Limit           19958 non-null  float64
 5   Number_of_Vehicles    19958 non-null  float64
 6   Driver_Alcohol        19958 non-null  float64
 7   Accident_Severity     19958 non-null  object 
 8   Road_Condition        19958 non-null  object 
 9   Vehicle_Type          19958 non-null  object 
 10  Driver_Age            19958 non-null  float64
 11  Driver_Experience     19958 non-null  float64
 12  Road_Light_Condition  19958 non-null  object 
 13  Accident              20000 non-null  int64  
dtypes: float64(6), int64(1), object(7)
memory usage: 2.1+ MB


In [162]:
df.describe(include="all")

,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
count,19958,19958,19958,19958.000000,19958.000000,19958.000000,19958.000000,19958,19958,19958,19958.000000,19958.000000,19958,20000.000000
unique,5,4,4,NaN,NaN,NaN,NaN,3,4,4,NaN,NaN,3,NaN
top,Clear,Highway,Afternoon,NaN,NaN,NaN,NaN,Low,Dry,Car,NaN,NaN,Artificial Light,NaN
freq,8366,10116,6826,NaN,NaN,NaN,NaN,11862,10072,14805,NaN,NaN,9978,NaN
mean,NaN,NaN,NaN,1.010923,71.448943,3.282694,0.164445,NaN,NaN,NaN,43.146758,38.859154,NaN,0.292000
std,NaN,NaN,NaN,0.783966,32.366260,1.999111,0.370688,NaN,NaN,NaN,15.099349,15.249536,NaN,0.454694
min,NaN,NaN,NaN,0.000000,30.000000,1.000000,0.000000,NaN,NaN,NaN,18.000000,9.000000,NaN,0.000000
25%,NaN,NaN,NaN,0.000000,50.000000,2.000000,0.000000,NaN,NaN,NaN,30.000000,26.000000,NaN,0.000000
50%,NaN,NaN,NaN,1.000000,60.000000,3.000000,0.000000,NaN,NaN,NaN,43.000000,39.000000,NaN,0.000000
75%,NaN,NaN,NaN,2.000000,80.000000,4.000000,0.000000,NaN,NaN,NaN,56.000000,52.000000,NaN,1.000000


In [163]:
df['Accident_Severity'].value_counts()  #Target variable

,count
Accident_Severity,
Low,11862
Moderate,6090
High,2006


In [164]:
df.isnull().sum()   #cheacking the null value

,0
Weather,42
Road_Type,42
Time_of_Day,42
Traffic_Density,42
Speed_Limit,42
Number_of_Vehicles,42
Driver_Alcohol,42
Accident_Severity,42
Road_Condition,42
Vehicle_Type,42


In [165]:
#Separating the numerical and categorical columns
num_col=df.select_dtypes(include=['int64','float64']).columns
cat_col=df.select_dtypes(include=['object']).columns

In [166]:
# Fillig the missing values
df[num_col] = df[num_col].fillna(df[num_col].median())
df[cat_col] = df[cat_col].fillna(df[cat_col].mode().iloc[0])
print("Numerical Coulumns", df[num_col].columns)
print("Categorical Columns", df[cat_col].columns)
df.isnull().sum()

Numerical Coulumns Index(['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles',
       'Driver_Alcohol', 'Driver_Age', 'Driver_Experience', 'Accident'],
      dtype='object')
Categorical Columns Index(['Weather', 'Road_Type', 'Time_of_Day', 'Accident_Severity',
       'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition'],
      dtype='object')


,0
Weather,0
Road_Type,0
Time_of_Day,0
Traffic_Density,0
Speed_Limit,0
Number_of_Vehicles,0
Driver_Alcohol,0
Accident_Severity,0
Road_Condition,0
Vehicle_Type,0


**Feature and target separation**

In [167]:
x=df.drop(columns=['Accident_Severity'], axis=1)
y=df['Accident_Severity']
print(x.shape)
print(y.shape)
x.columns


(20000, 13)
(20000,)


Index(['Weather', 'Road_Type', 'Time_of_Day', 'Traffic_Density', 'Speed_Limit',
       'Number_of_Vehicles', 'Driver_Alcohol', 'Road_Condition',
       'Vehicle_Type', 'Driver_Age', 'Driver_Experience',
       'Road_Light_Condition', 'Accident'],
      dtype='object')

In [168]:
y.count()

np.int64(20000)

In [169]:

print("Clases of Target Variable:", y.value_counts())


Clases of Target Variable: Accident_Severity
Low         11904
Moderate     6090
High         2006
Name: count, dtype: int64


In [170]:
order = {'Low': 0, 'Moderate': 1, 'High': 2}
y = y.map(order)
y

,Accident_Severity
0,0
1,1
2,0
3,0
4,0
...,...
19995,1
19996,1
19997,1
19998,0


In [171]:
#get the numerical features and categorical features without Target variable
numerical_features = x.select_dtypes(include=['int64', 'float64']).columns
categorical_features = x.select_dtypes(include=['object']).columns
print("Numerical Features:", numerical_features)
print("Categorical Features:", categorical_features)

Numerical Features: Index(['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles',
       'Driver_Alcohol', 'Driver_Age', 'Driver_Experience', 'Accident'],
      dtype='object')
Categorical Features: Index(['Weather', 'Road_Type', 'Time_of_Day', 'Road_Condition', 'Vehicle_Type',
       'Road_Light_Condition'],
      dtype='object')


In [172]:
# Apply different processing step to different columns group
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])
categorical_features


Index(['Weather', 'Road_Type', 'Time_of_Day', 'Road_Condition', 'Vehicle_Type',
       'Road_Light_Condition'],
      dtype='object')

**Split and Train the data**

In [173]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42,  stratify=y
)

print("Shape of x_train:", x_train.shape)
print("Shape of x_test:", x_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)
y_train.value_counts()

Shape of x_train: (16000, 13)
Shape of x_test: (4000, 13)
Shape of y_train: (16000,)
Shape of y_test: (4000,)


,count
Accident_Severity,
0,9523
1,4872
2,1605


**Applying Models**

In [174]:
#Logistic regression
lr = Pipeline(steps=[('preprocessor', preprocessor),
                     ('classifier', LogisticRegression(max_iter=1000))])

lr=lr.fit(x_train, y_train)
y_pred = lr.predict(x_test)

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy:\n", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Confusion Matrix:
 [[2381    0    0]
 [1218    0    0]
 [ 401    0    0]]
accuracy:
 0.59525
Classification Report:
               precision    recall  f1-score   support

           0       0.60      1.00      0.75      2381
           1       0.00      0.00      0.00      1218
           2       0.00      0.00      0.00       401

    accuracy                           0.60      4000
   macro avg       0.20      0.33      0.25      4000
weighted avg       0.35      0.60      0.44      4000



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [175]:
#Random Forest

rf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('model', RandomForestClassifier(
                          class_weight='balanced_subsample',
                          n_estimators=300,
                          max_depth=None,
                          min_samples_split=5,
                          min_samples_leaf=2,
                          random_state=42,

                      ))])

rf = rf.fit(x_train, y_train)
y_pred = rf.predict(x_test)

print("confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy:\n", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))


confusion Matrix:
 [[2101  268   12]
 [1106  107    5]
 [ 358   40    3]]
accuracy:
 0.55275
Classification Report:
               precision    recall  f1-score   support

           0       0.59      0.88      0.71      2381
           1       0.26      0.09      0.13      1218
           2       0.15      0.01      0.01       401

    accuracy                           0.55      4000
   macro avg       0.33      0.33      0.28      4000
weighted avg       0.44      0.55      0.46      4000



**Compute class weights**

In [176]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight='balanced', classes=classes, y=y_train
)

weight_map = dict(zip(classes, class_weights))
sample_weights = y_train.map(weight_map)
sample_weights

,Accident_Severity
10328,1.094691
2945,1.094691
6774,0.560048
15521,1.094691
18340,1.094691
...,...
494,3.322949
7893,0.560048
3907,0.560048
8269,1.094691


In [177]:
# XGBoost Model
xgb = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', XGBClassifier(
                          objective='multi:softmax',
                         num_class=3,
                          eval_metric='mlogloss',
                          max_depth=5,
                          sub_sample=0.8,
                          colsample_bytree=0.8,

                          n_estimators=300,
                          learning_rate=0.05
                      ))])

xgb.fit(x_train, y_train, classifier__sample_weight=sample_weights)
y_pred = xgb.predict(x_test)
y_prob = xgb.predict_proba(x_test)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("accuracy:\n", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:39:47] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "sub_sample" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Confusion Matrix:
 [[937 810 634]
 [477 439 302]
 [138 147 116]]
accuracy:
 0.373
Classification Report:
               precision    recall  f1-score   support

           0       0.60      0.39      0.48      2381
           1       0.31      0.36      0.34      1218
           2       0.11      0.29      0.16       401

    accuracy                           0.37      4000
   macro avg       0.34      0.35      0.32      4000
weighted avg       0.47      0.37      0.40      4000



**Applying threshold**

In [178]:
for t in [0.4, 0.5, 0.6, 0.7]:
    preds = np.argmax(y_prob / [1, 1, t], axis=1)
    print(f"\nThreshold = {t}")
    print(accuracy_score(y_test, preds))
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))



Threshold = 0.4
0.12925
[[  99   67 2215]
 [  47   45 1126]
 [  19    9  373]]
              precision    recall  f1-score   support

           0       0.60      0.04      0.08      2381
           1       0.37      0.04      0.07      1218
           2       0.10      0.93      0.18       401

    accuracy                           0.13      4000
   macro avg       0.36      0.34      0.11      4000
weighted avg       0.48      0.13      0.08      4000


Threshold = 0.5
0.1565
[[ 193  128 2060]
 [  97   89 1032]
 [  32   25  344]]
              precision    recall  f1-score   support

           0       0.60      0.08      0.14      2381
           1       0.37      0.07      0.12      1218
           2       0.10      0.86      0.18       401

    accuracy                           0.16      4000
   macro avg       0.36      0.34      0.15      4000
weighted avg       0.48      0.16      0.14      4000


Threshold = 0.6
0.1835
[[ 305  215 1861]
 [ 167  131  920]
 [  58   45  298]]


In [180]:
FINAL_THRESHOLD = 0.7

y_pred_final = np.argmax(y_prob / [1, 1, FINAL_THRESHOLD], axis=1)

print(confusion_matrix(y_test, y_pred_final))
print(accuracy_score(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final))

[[ 444  336 1601]
 [ 238  200  780]
 [  81   73  247]]
0.22275
              precision    recall  f1-score   support

           0       0.58      0.19      0.28      2381
           1       0.33      0.16      0.22      1218
           2       0.09      0.62      0.16       401

    accuracy                           0.22      4000
   macro avg       0.33      0.32      0.22      4000
weighted avg       0.46      0.22      0.25      4000

